# Segmentation du Risque — Classification par Arbre de Décision

**Objectif** : Prédire le niveau de risque (`Risk_Level` : Low / Medium / High) des tâches de gestion de projet.  
**Méthode** : Decision Tree Classifier (scikit-learn)  
**Méthodologie** : Chapitre 2 — Préparation des données & TP Heart Disease Prediction

---
## Étape 1 — Chargement et Inspection

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

In [ ]:
df = pd.read_csv('Project-Management-2-enriched.csv')

print('='*60)
print('INFO DU DATASET')
print('='*60)
print(df.info())
print('\n' + '='*60)
print('APERÇU (5 premières lignes)')
print('='*60)
print(df.head())
print('\n' + '='*60)
print('STATISTIQUES DESCRIPTIVES')
print('='*60)
print(df.describe())
print('\n' + '='*60)
print('VALEURS MANQUANTES PAR COLONNE')
print('='*60)
print(df.isnull().sum())
print('\n' + '='*60)
print('NOMBRE DE DOUBLONS')
print('='*60)
print(df.duplicated().sum())

In [ ]:
print('Dimensions du dataset :', df.shape)
print(f'\nColonne cible identifiée : Risk_Level')
print(f'Distribution de la cible :\n{df["Risk_Level"].value_counts()}')
print(f'\nProportion (%) :\n{df["Risk_Level"].value_counts(normalize=True)*100:.2f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = {'Low': '#2ecc71', 'Medium': '#f39c12', 'High': '#e74c3c'}
counts = df['Risk_Level'].value_counts()
bars = ax.bar(counts.index, counts.values, color=[colors[c] for c in counts.index], edgecolor='black')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, str(val),
            ha='center', va='bottom', fontweight='bold', fontsize=13)
ax.set_title('Distribution de la variable cible — Risk_Level', fontsize=14, fontweight='bold')
ax.set_ylabel('Nombre de tâches')
ax.set_xlabel('Niveau de risque')
plt.tight_layout()
plt.show()

---
## Étape 2 — Nettoyage des données

In [ ]:
initial_rows = len(df)
print(f'Nombre de lignes initial : {initial_rows}')

# --- 2.1 Suppression des doublons ---
n_dup = df.duplicated().sum()
df = df.drop_duplicates()
print(f'Doublons supprimés : {n_dup}')
print(f'Lignes après suppression des doublons : {len(df)}')

In [ ]:
# --- 2.2 Gestion des valeurs manquantes ---
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_report = pd.DataFrame({'NaN_count': missing, 'Pourcentage(%)': missing_pct})
print('Rapport des valeurs manquantes :')
print(missing_report[missing_report['NaN_count'] > 0])

if missing.sum() == 0:
    print('\n✅ Aucune valeur manquante détectée.')
else:
    pct_rows_with_nan = (df.isnull().any(axis=1).sum() / len(df)) * 100
    if pct_rows_with_nan < 5:
        print(f'\n< 5% de lignes avec NaN ({pct_rows_with_nan:.2f}%) → suppression de ces lignes.')
        df = df.dropna()
    else:
        print(f'\n≥ 5% de lignes avec NaN ({pct_rows_with_nan:.2f}%) → imputation.')
        for col in df.columns:
            if df[col].isnull().sum() > 0:
                if df[col].dtype in ['float64', 'int64']:
                    df[col].fillna(df[col].median(), inplace=True)
                    print(f'  • {col} → imputation par la médiane')
                else:
                    df[col].fillna(df[col].mode()[0], inplace=True)
                    print(f'  • {col} → imputation par le mode')

print(f'\nLignes après gestion des NaN : {len(df)}')

In [ ]:
# --- 2.3 Suppression des colonnes identifiantes / non pertinentes ---
cols_to_drop = []
drop_reasons = {}

id_cols = ['Project ID', 'Task ID', 'Project Name', 'Task Name', 'Start Date', 'End Date']
for col in id_cols:
    if col in df.columns:
        cols_to_drop.append(col)
        drop_reasons[col] = 'Identifiant / texte libre / date brute'

print('Colonnes identifiantes / non pertinentes supprimées :')
for c in cols_to_drop:
    print(f'  ✗ {c} → {drop_reasons[c]}')

df = df.drop(columns=cols_to_drop)

In [ ]:
# --- 2.4 Suppression des colonnes constantes ---
const_cols = [col for col in df.columns if df[col].nunique() <= 1]
if const_cols:
    print('Colonnes constantes supprimées :')
    for c in const_cols:
        print(f'  ✗ {c}')
        drop_reasons[c] = 'Colonne constante (1 seule valeur unique)'
    df = df.drop(columns=const_cols)
else:
    print('✅ Aucune colonne constante détectée.')

In [ ]:
# --- 2.5 Suppression des colonnes catégorielles à >10 modalités ---
cat_cols = df.select_dtypes(include='object').columns.tolist()
if 'Risk_Level' in cat_cols:
    cat_cols.remove('Risk_Level')

high_card_cols = []
print('Analyse des modalités des colonnes catégorielles :')
for col in cat_cols:
    n_unique = df[col].nunique()
    status = '✅' if n_unique <= 10 else '✗ SUPPRIMÉE (>10 modalités)'
    print(f'  {col} : {n_unique} modalités → {status}')
    if n_unique > 10:
        high_card_cols.append(col)
        drop_reasons[col] = f'Catégorielle à trop de modalités ({n_unique} > 10)'

if high_card_cols:
    df = df.drop(columns=high_card_cols)
    print(f'\nColonnes supprimées pour cardinalité élevée : {high_card_cols}')
else:
    print('\n✅ Toutes les colonnes catégorielles ont ≤10 modalités.')

In [ ]:
# --- 2.6 Suppression des colonnes fortement corrélées (|r| > 0.95) ---
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df[num_cols].corr().abs()

upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_pairs = []
corr_cols_to_drop = set()

for col in upper.columns:
    for idx in upper.index:
        if upper.loc[idx, col] > 0.95:
            high_corr_pairs.append((idx, col, upper.loc[idx, col]))
            corr_with_target_idx = df[idx].corr(df['Risk_Level'].map({'Low': 0, 'Medium': 1, 'High': 2}).astype(float))
            corr_with_target_col = df[col].corr(df['Risk_Level'].map({'Low': 0, 'Medium': 1, 'High': 2}).astype(float))
            to_remove = col if abs(corr_with_target_idx) >= abs(corr_with_target_col) else idx
            corr_cols_to_drop.add(to_remove)

if high_corr_pairs:
    print('Paires fortement corrélées (|r| > 0.95) :')
    for a, b, r in high_corr_pairs:
        print(f'  • {a} ↔ {b} : r = {r:.4f}')
    print(f'\nColonnes supprimées (moins informatives pour la cible) : {corr_cols_to_drop}')
    for c in corr_cols_to_drop:
        drop_reasons[c] = 'Fortement corrélée (|r| > 0.95)'
    df = df.drop(columns=list(corr_cols_to_drop))
else:
    print('✅ Aucune paire de colonnes avec |corrélation| > 0.95.')

print(f'\nColonnes restantes ({len(df.columns)}) : {list(df.columns)}')

In [ ]:
# Matrice de corrélation après nettoyage
num_cols_clean = df.select_dtypes(include=[np.number]).columns.tolist()
if len(num_cols_clean) > 1:
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(df[num_cols_clean].corr(), annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, square=True, linewidths=0.5, ax=ax)
    ax.set_title('Matrice de corrélation (après nettoyage)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## Étape 3 — Encodage des variables catégorielles

In [ ]:
# --- 3.1 Encodage de la cible avec LabelEncoder ---
le_target = LabelEncoder()
df['Risk_Level_encoded'] = le_target.fit_transform(df['Risk_Level'])

print('Encodage de la cible (Risk_Level) :')
for cls, code in zip(le_target.classes_, le_target.transform(le_target.classes_)):
    print(f'  {cls} → {code}')

df = df.drop(columns=['Risk_Level'])
df = df.rename(columns={'Risk_Level_encoded': 'Risk_Level'})

In [ ]:
# --- 3.2 Encodage ordinal : Priority (Low < Medium < High) ---
priority_map = {'Low': 0, 'Medium': 1, 'High': 2}
if 'Priority' in df.columns:
    df['Priority'] = df['Priority'].map(priority_map)
    print('Encodage ordinal de Priority :')
    for k, v in priority_map.items():
        print(f'  {k} → {v}')

In [ ]:
# --- 3.3 One-Hot Encoding des variables nominales (≤10 modalités) ---
cat_cols_remaining = df.select_dtypes(include='object').columns.tolist()
print(f'Colonnes catégorielles à encoder en One-Hot : {cat_cols_remaining}')

for col in cat_cols_remaining:
    print(f'  • {col} : {df[col].nunique()} modalités → {list(df[col].unique())}')

df = pd.get_dummies(df, columns=cat_cols_remaining, drop_first=False)

print(f'\nDimensions après encodage : {df.shape}')
print(f'Colonnes : {list(df.columns)}')

In [ ]:
# Vérification : pas de scaling nécessaire pour les arbres de décision
print('⚠️ Aucun scaling appliqué — les arbres de décision sont invariants aux échelles.')
print(f'\nAperçu du dataset final :')
df.head()

---
## Étape 4 — Séparation features / cible

In [ ]:
X = df.drop(columns=['Risk_Level'])
y = df['Risk_Level']

print(f'Features (X) : {X.shape[1]} colonnes')
print(f'Cible (y)    : {y.shape[0]} échantillons')
print(f'\nListe des features :')
for i, col in enumerate(X.columns, 1):
    print(f'  {i:2d}. {col}')

---
## Étape 5 — Division train / test (80% / 20%)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Ensemble d\'entraînement : {X_train.shape[0]} lignes ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Ensemble de test         : {X_test.shape[0]} lignes ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'\nDistribution de y_train :\n{y_train.value_counts().sort_index()}')
print(f'\nDistribution de y_test  :\n{y_test.value_counts().sort_index()}')

---
## Étape 6 — Entraînement du Decision Tree

In [ ]:
model = DecisionTreeClassifier(random_state=42)
model.fit(X_train, y_train)

print(f'Arbre entraîné avec succès.')
print(f'Profondeur de l\'arbre     : {model.get_depth()}')
print(f'Nombre de feuilles         : {model.get_n_leaves()}')
print(f'Nombre de features utilisées : {model.n_features_in_}')

In [ ]:
# Visualisation de l'arbre (limité à profondeur 4 pour lisibilité)
fig, ax = plt.subplots(figsize=(28, 12))
plot_tree(model, max_depth=4, feature_names=X.columns.tolist(),
          class_names=le_target.classes_.tolist(),
          filled=True, rounded=True, fontsize=8, ax=ax,
          proportion=True)
ax.set_title('Arbre de Décision — Risk Level (profondeur affichée : 4)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Texte de l'arbre (limité à profondeur 3)
print('Règles de l\'arbre (profondeur max affichée = 3) :')
print(export_text(model, feature_names=X.columns.tolist(), max_depth=3))

In [ ]:
# Importance des features
feat_imp = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=True)
feat_imp_nonzero = feat_imp[feat_imp > 0]

fig, ax = plt.subplots(figsize=(10, max(6, len(feat_imp_nonzero)*0.4)))
feat_imp_nonzero.plot(kind='barh', color='#3498db', edgecolor='black', ax=ax)
ax.set_title('Importance des features — Decision Tree', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance (Gini)')
plt.tight_layout()
plt.show()

---
## Étape 7 — Évaluation du modèle

In [ ]:
y_pred = model.predict(X_test)

print('='*60)
print('ÉVALUATION SUR L\'ENSEMBLE DE TEST')
print('='*60)
print(f'\nAccuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'\n--- Matrice de confusion ---')
print(confusion_matrix(y_test, y_pred))
print(f'\n--- Rapport de classification ---')
print(classification_report(y_test, y_pred, target_names=le_target.classes_))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le_target.classes_)
disp.plot(cmap='Blues', ax=ax, values_format='d')
ax.set_title('Matrice de Confusion — Decision Tree (sans élagage)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Performance sur train vs test (détection de surapprentissage)
y_train_pred = model.predict(X_train)
train_acc = accuracy_score(y_train, y_train_pred)
test_acc = accuracy_score(y_test, y_pred)

print(f'Accuracy Train : {train_acc:.4f}')
print(f'Accuracy Test  : {test_acc:.4f}')
print(f'Écart           : {train_acc - test_acc:.4f}')

if train_acc - test_acc > 0.10:
    print('\n⚠️ Surapprentissage probable (écart > 10%) → élagage recommandé (Étape 8).')
else:
    print('\n✅ Pas de surapprentissage significatif détecté.')

---
## Étape 8 — Élagage (Pruning) pour éviter le surapprentissage

### 8.1 — Recherche de la profondeur optimale (max_depth)

In [ ]:
depths = range(2, 16)
train_scores = []
test_scores = []

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train, y_train)
    train_scores.append(accuracy_score(y_train, dt.predict(X_train)))
    test_scores.append(accuracy_score(y_test, dt.predict(X_test)))

best_depth = depths[np.argmax(test_scores)]
best_test_acc = max(test_scores)

print(f'Meilleure profondeur : {best_depth} (Accuracy test = {best_test_acc:.4f})')

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(list(depths), train_scores, 'o-', label='Train Accuracy', color='#3498db', linewidth=2)
ax.plot(list(depths), test_scores, 's-', label='Test Accuracy', color='#e74c3c', linewidth=2)
ax.axvline(x=best_depth, color='green', linestyle='--', alpha=0.7, label=f'Best depth = {best_depth}')
ax.set_xlabel('max_depth', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Accuracy Train vs Test en fonction de max_depth', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 8.2 — Pruning par Cost-Complexity (ccp_alpha)

In [ ]:
path = model.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas
impurities = path.impurities

train_scores_ccp = []
test_scores_ccp = []

for alpha in ccp_alphas:
    dt = DecisionTreeClassifier(ccp_alpha=alpha, random_state=42)
    dt.fit(X_train, y_train)
    train_scores_ccp.append(accuracy_score(y_train, dt.predict(X_train)))
    test_scores_ccp.append(accuracy_score(y_test, dt.predict(X_test)))

best_alpha_idx = np.argmax(test_scores_ccp)
best_alpha = ccp_alphas[best_alpha_idx]
best_alpha_acc = test_scores_ccp[best_alpha_idx]

print(f'Meilleur ccp_alpha : {best_alpha:.6f} (Accuracy test = {best_alpha_acc:.4f})')

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(ccp_alphas, train_scores_ccp, 'o-', label='Train Accuracy', color='#3498db', markersize=3, linewidth=1)
ax.plot(ccp_alphas, test_scores_ccp, 's-', label='Test Accuracy', color='#e74c3c', markersize=3, linewidth=1)
ax.axvline(x=best_alpha, color='green', linestyle='--', alpha=0.7, label=f'Best α = {best_alpha:.4f}')
ax.set_xlabel('ccp_alpha', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Accuracy Train vs Test en fonction de ccp_alpha', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Modèle final élagué ---
model_pruned = DecisionTreeClassifier(
    max_depth=best_depth,
    ccp_alpha=best_alpha,
    random_state=42
)
model_pruned.fit(X_train, y_train)
y_pred_pruned = model_pruned.predict(X_test)

print('='*60)
print('MODÈLE FINAL ÉLAGUÉ')
print('='*60)
print(f'max_depth  = {best_depth}')
print(f'ccp_alpha  = {best_alpha:.6f}')
print(f'Profondeur effective : {model_pruned.get_depth()}')
print(f'Nombre de feuilles   : {model_pruned.get_n_leaves()}')
print(f'\nAccuracy : {accuracy_score(y_test, y_pred_pruned):.4f}')
print(f'\n--- Matrice de confusion ---')
print(confusion_matrix(y_test, y_pred_pruned))
print(f'\n--- Rapport de classification ---')
print(classification_report(y_test, y_pred_pruned, target_names=le_target.classes_))

In [ ]:
# Matrice de confusion du modèle élagué
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

cm1 = confusion_matrix(y_test, y_pred)
disp1 = ConfusionMatrixDisplay(confusion_matrix=cm1, display_labels=le_target.classes_)
disp1.plot(cmap='Oranges', ax=axes[0], values_format='d')
axes[0].set_title('Sans élagage', fontsize=13, fontweight='bold')

cm2 = confusion_matrix(y_test, y_pred_pruned)
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm2, display_labels=le_target.classes_)
disp2.plot(cmap='Greens', ax=axes[1], values_format='d')
axes[1].set_title('Avec élagage (pruned)', fontsize=13, fontweight='bold')

plt.suptitle('Comparaison des matrices de confusion', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Arbre élagué complet
fig, ax = plt.subplots(figsize=(28, 12))
plot_tree(model_pruned, feature_names=X.columns.tolist(),
          class_names=le_target.classes_.tolist(),
          filled=True, rounded=True, fontsize=8, ax=ax,
          proportion=True)
ax.set_title(f'Arbre de Décision élagué (depth={model_pruned.get_depth()}, leaves={model_pruned.get_n_leaves()})',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Importance des features (modèle élagué)
feat_imp_pruned = pd.Series(model_pruned.feature_importances_, index=X.columns).sort_values(ascending=True)
feat_imp_pruned_nz = feat_imp_pruned[feat_imp_pruned > 0]

fig, ax = plt.subplots(figsize=(10, max(6, len(feat_imp_pruned_nz)*0.4)))
feat_imp_pruned_nz.plot(kind='barh', color='#2ecc71', edgecolor='black', ax=ax)
ax.set_title('Importance des features — Arbre élagué', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance (Gini)')
plt.tight_layout()
plt.show()

---
## Étape 9 — Résumé des actions

In [ ]:
final_acc = accuracy_score(y_test, y_pred_pruned)
report = classification_report(y_test, y_pred_pruned, target_names=le_target.classes_, output_dict=True)
f1_macro = report['macro avg']['f1-score']
f1_weighted = report['weighted avg']['f1-score']

original_acc = accuracy_score(y_test, y_pred)

print(f"""
{'='*70}
                    RÉSUMÉ DU PIPELINE
{'='*70}

📊 DONNÉES
  • Lignes initiales              : {initial_rows}
  • Lignes après nettoyage        : {len(df)}
  • Doublons supprimés            : {n_dup}

🗑️ COLONNES SUPPRIMÉES
""")

for col, reason in drop_reasons.items():
    print(f'  ✗ {col:40s} → {reason}')

print(f"""
🔧 IMPUTATION
  • Méthode : {'Aucune (pas de valeurs manquantes)' if missing.sum() == 0 else 'Médiane (num) / Mode (cat)'}

🔤 ENCODAGE
  • Cible (Risk_Level)     : LabelEncoder (High=0, Low=1, Medium=2)
  • Priority               : Encodage ordinal (Low=0, Medium=1, High=2)
  • Nominales restantes    : One-Hot Encoding (drop_first=False)
  • Scaling                : Non appliqué (Decision Tree invariant aux échelles)

🌳 MODÈLE — DECISION TREE
  • Arbre sans élagage :
      - Profondeur       : {model.get_depth()}
      - Feuilles         : {model.get_n_leaves()}
      - Accuracy (test)  : {original_acc:.4f}

  • Arbre élagué (final) :
      - max_depth         : {best_depth}
      - ccp_alpha          : {best_alpha:.6f}
      - Profondeur effective : {model_pruned.get_depth()}
      - Feuilles           : {model_pruned.get_n_leaves()}
      - Accuracy (test)    : {final_acc:.4f}
      - F1-score (macro)   : {f1_macro:.4f}
      - F1-score (weighted): {f1_weighted:.4f}

📌 REMARQUE
  Les arbres de décision sont robustes aux variables numériques et
  catégorielles encodées, mais sensibles au surapprentissage.
  L'élagage (max_depth + ccp_alpha) a été appliqué pour régulariser.
{'='*70}
""")